# Comparing Merge, Rebase, and Squash Strategies

This notebook provides a reference comparison of three fundamental Git integration strategies: merge, rebase, and squash. Each strategy reshapes commit history differently, and the choice depends on the workflow, team preferences, and the nature of the project.

## Purpose

The purpose of this notebook is to document and compare the three primary ways Git integrates separate lines of development. Understanding the differences is essential for maintaining a clean, readable project history.

### When to Use Merge

- Use merge when the complete history of a feature branch should be preserved, including the branch point. Merge creates a merge commit that records the two branch histories, making it easy to trace when a feature was integrated.
- Merge is preferred for long-lived feature branches or when auditability of the integration point is important.
- It is the safest choice when the branch has diverged significantly and both histories should be kept intact.

In [ ]:
# Setup: create a temporary repo for demonstration
import subprocess
import os

repo_dir = '/tmp/git-demonstrator'
if os.path.exists(repo_dir):
    import shutil
    shutil.rmtree(repo_dir)

subprocess.run(['git', 'init'], cwd=repo_dir, check=True)
subprocess.run(['git', 'config', 'user.name', 'Test User'], cwd=repo_dir, check=True)
subprocess.run(['git', 'config', 'user.email', 'test@example.com'], cwd=repo_dir, check=True)

# Initial commit to establish a base
subprocess.run(['git', 'commit', '--allow-empty', '-m', 'initial'], cwd=repo_dir, check=True)

# Create a feature branch and make a commit
subprocess.run(['git', 'checkout', '-b', 'feature-branch'], cwd=repo_dir, check=True)
subprocess.run(['git', 'commit', '--allow-empty', '-m', 'feature work'], cwd=repo_dir, check=True)

# Merge the feature branch into main, preserving the branch history
subprocess.run(['git', 'checkout', 'main'], cwd=repo_dir, check=True)
subprocess.run(['git', 'merge', 'feature-branch'], cwd=repo_dir, check=True)

# Demonstrate the merge result
subprocess.run(['git', 'log', '--oneline', '--graph'], cwd=repo_dir, check=True)

### When to Use Rebase

- Use rebase when a linear project history is desired. Rebase reapplies commits from one branch onto another, rewriting commit hases. This results in a clean, linear history without merge commits.
- Rebase is preferred for short-lived feature branches that are kept up-to-date with the main branch, especially before a pull request merge.
- Caution: Rebasing public branches rewrites shared history and can cause conflicts for other collaborators. Never rebase branches that exist remotely unless the implications are fully understood.

In [ ]:
# Demonstrate git rebase
subprocess.run(['git', 'checkout', 'main'], cwd=repo_dir, check=True)
subprocess.run(['git', 'checkout', '-b', 'feature-branch2'], cwd=repo_dir, check=True)
subprocess.run(['git', 'commit', '--allow-empty', '-m', 'feature commit 1'], cwd=repo_dir, check=True)
subprocess.run(['git', 'commit', '--allow-empty', '-m', 'feature commit 2'], cwd=repo_dir, check=True)

# Main has progressed; rebase feature-branch2 onto main
subprocess.run(['git', 'checkout', 'feature-branch2'], cwd=repo_dir, check=True)
subprocess.run(['git', 'rebase', 'main'], cwd=repo_dir, check=True)

# Result: linear history
subprocess.run(['git', 'log', '--oneline', '--graph'], cwd=repo_dir, check=True)

### What is Squash?

- Use `git merge --squash` when the changes from a feature branch should be integrated as a single commit on the target branch. Unlike a regular merge, squash does NOT create a merge commit record of the branch relationship — it simply applies the changes as one new commit.
- Squash is useful when feature changes should be integrated without preserving the individual commit history from the feature branch.
- After `git merge --squash`, a final `git commit` must be run to complete the squash commit.
- Squash is NOT the same as `git rebase --squash`; the latter is interactive and part of a rebase operation.

In [ ]:
# Demonstrate git merge --squash
subprocess.run(['git', 'checkout', 'main'], cwd=repo_dir, check=True)
subprocess.run(['git', 'checkout', '-b', 'feature-branch3'], cwd=repo_dir, check=True)
subprocess.run(['git', 'commit', '--allow-empty', '-m', 'initial'], cwd=repo_dir, check=True)
subprocess.run(['git', 'commit', '--allow-empty', '-m', 'feature commit A'], cwd=repo_dir, check=True)
subprocess.run(['git', 'commit', '--allow-empty', '-m', 'feature commit B'], cwd=repo_dir, check=True)

# Merge with squash — stages all changes from the branch without creating a merge commit
subprocess.run(['git', 'merge', '--squash', 'feature-branch3'], cwd=repo_dir, check=True)

# Commit the squash manually
subprocess.run(['git', 'commit', '-m', 'squashed feature integration'], cwd=repo_dir, check=True)

# Verify the history
subprocess.run(['git', 'log', '--oneline', '--graph'], cwd=repo_dir, check=True)

### Verification

After integrating a feature, verify the result with:
- `git log --oneline --graph` — view the commit history tree
- `git diff main..feature-branch` — check what differs between branches
- `git status` — ensure the working tree is clean
- Review the commit messages for clarity and consistency.

### Common Errors and Gotchas

- Confusing merge and rebase: Merge preserves history; rebase rewrites it. Using the wrong strategy can lead to unexpected commit duplication or loss.
- Rebasing shared branches: Rebasing a branch that others have based work on will rewrite history and cause merge conflicts. Always work on local, private branches when rebasing.
- Forgetting to commit after squash: `git merge --squash` stages changes but does NOT commit them. Running `git commit` separately is required.
- Fast-forward merges: By default, `git merge` may fast-forward if the target branch has not diverged. Use `git merge --no-ff` to always create a merge commit for traceability.

### References

- [Pro Git Book: Basic Branching and Merging](https://git-scm.com/book/en/v2/Git-Branching-and-merging)
- [Git Documentation: git-merge](https://git-scm.com/docs/git-merge)
- [Git Documentation: git-rebase](https://git-scm.com/docs/git-rebase)